In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les prérequis suivants.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

Le package d'utilitaires pour les addons Qiskit est un ensemble de fonctionnalités qui viennent compléter les workflows faisant appel à un ou plusieurs addons Qiskit. Il contient par exemple des fonctions pour créer des hamiltoniens, générer des circuits d'évolution temporelle de Trotter, et découper ou recombiner des circuits quantiques.

## Installation

Il existe deux façons d'installer les utilitaires pour les addons Qiskit : via PyPI ou en compilant depuis les sources. Il est recommandé d'installer ces packages dans un [environnement virtuel](https://docs.python.org/3.10/tutorial/venv.html) afin d'isoler les dépendances.

### Installer depuis PyPI

La façon la plus simple d'installer le package d'utilitaires pour les addons Qiskit est via PyPI.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### Installer depuis les sources
<details>
<summary>
Clique ici pour savoir comment installer ce package manuellement.
</summary>

Si tu souhaites contribuer à ce package ou l'installer manuellement, commence par cloner le dépôt :

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

puis installe le package via `pip`. Si tu prévois d'exécuter les tutoriels inclus dans le dépôt, installe également les dépendances notebook. Si tu prévois de contribuer au développement, installe les dépendances `dev`.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## Premiers pas avec les utilitaires
Le package `qiskit-addon-utils` comprend plusieurs modules, notamment un pour la génération de problèmes liés à la simulation de systèmes quantiques, un pour la coloration de graphes permettant de placer les portes dans un circuit quantique de manière plus efficace, et un pour le découpage de circuits, utile notamment pour la [rétropropagation d'opérateurs](/guides/qiskit-addons-obp). Les sections suivantes résument chaque module. La [documentation de l'API](https://docs.quantum.ibm.com/api/qiskit-addon-utils) du package contient également des informations utiles.

### Génération de problèmes
Le contenu du module [`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) comprend :

- Une fonction [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian), qui génère une représentation `SparsePauliOp` d'un modèle XYZ de type Ising tenant compte de la connectivité :

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- Une fonction [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit), qui construit un circuit modélisant l'évolution temporelle d'un opérateur donné.
- Trois objets [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy) permettant d'énumérer différents ordres de chaînes de Pauli. Cela est particulièrement utile en combinaison avec la coloration de graphes, et ces objets peuvent être passés en argument à `generate_xyz_hamiltonian()` et à `generate_time_evolution_circuit()`.

### Coloration de graphes
Le module [`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) sert à colorier les arêtes d'une carte de couplage et à exploiter ce coloriage pour placer les portes dans un circuit quantique de façon plus efficace. L'objectif de cette carte de couplage à arêtes colorées est de trouver un ensemble de couleurs tel qu'aucune paire d'arêtes de la même couleur ne partage un nœud commun. Sur un QPU, cela signifie que les portes situées sur des arêtes de même couleur (connexions entre qubits) peuvent être exécutées simultanément, ce qui accélère l'exécution du circuit.

Pour illustrer cela rapidement, tu peux utiliser la fonction [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges) pour générer un coloriage d'arêtes pour un circuit naïf qui applique une `CZGate` sur chaque connexion de qubit. L'extrait de code ci-dessous utilise la carte de couplage du backend `FakeSherbrooke`, crée ce circuit naïf, puis utilise `auto_color_edges()` pour générer un circuit équivalent plus efficace.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### Découpage
Enfin, le module [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) contient des fonctions et des passes du Transpiler pour travailler avec des « tranches » de circuit, c'est-à-dire des partitions temporelles d'un [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) couvrant tous les qubits. Ces tranches sont principalement utilisées pour la [rétropropagation d'opérateurs](/guides/qiskit-addons-obp-get-started). Les quatre principales façons de découper un circuit sont : par type de porte, par profondeur, par coloriage, ou par instructions [`Barrier`](../api/qiskit/circuit#barrier). Ces fonctions de découpage retournent une liste d'objets [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit). Les tranches peuvent également être recombinées avec la fonction [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices). Consulte la [référence de l'API](../api/qiskit-addon-utils/slicing) du module pour plus d'informations.

Voici quelques exemples illustrant comment créer ces tranches à partir du circuit suivant :

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Résultat de la cellule de code précédente](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

Lorsqu'il n'y a pas de structure évidente à exploiter dans un circuit pour la rétropropagation d'opérateurs, tu peux partitionner le circuit en tranches d'une profondeur donnée.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Résultat de la cellule de code précédente](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

Si tu as une stratégie de découpage personnalisée, tu peux placer des barrières dans le circuit pour indiquer les points de découpage, puis utiliser la fonction [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers).